# Wordfeud RL Environment

Pure-Python game simulator + Gymnasium env (agent = player 0 vs a greedy opponent).
The vision pipeline is *not* used here — this is the fast loop for training.

In [1]:
from src.dictionary import load_dictionary
from src.move_generator import WordfeudEngine
from src.env import WordfeudEnv
from src.eval import evaluate, best_candidate_policy

words = [w for lst in load_dictionary().values() for w in lst]
engine = WordfeudEngine(words)

# Static bonus board. Swap in a real layout (e.g. WordfeudMap(path).map.tolist()).
bonus = [[1] * 15 for _ in range(15)]
bonus[7][7] = 4  # double-word at centre

env = WordfeudEnv(engine, bonus, max_candidates=48, seed=0)

## One step
`info["moves"]` is the live candidate list; `action` indexes into it (last index = pass).

In [2]:
obs, info = env.reset(seed=0)
print(f"{len(info['moves'])} legal moves; top: {info['moves'][0]}")

obs, reward, done, trunc, info = env.step(0)  # play the highest-scoring move
print("reward:", reward, "| done:", done)
print(env.render())

48 legal moves; top: Move('begiv' H @(7, 7) score=28 tiles=5)
reward: 28.0 | done: False
P0=28 P1=67 | bag=79 | to_move=0 | done=False
. . . . . . . . . . . . . . .
. . . . . . . . . . . . . . .
. . . . . . . . . . . . . . .
. . . . . . . . . . . . k . .
. . . . . . . . . . . . n . .
. . . . . . . . . . . . o . .
. . . . . . . . . . . . f . .
. . . . . . . b e g i v e . .
. . . . . . . . . . . . d . .
. . . . . . . . . . . . t . .
. . . . . . . . . . . . . . .
. . . . . . . . . . . . . . .
. . . . . . . . . . . . . . .
. . . . . . . . . . . . . . .
. . . . . . . . . . . . . . .


## Baseline: always play the best-scoring move (= greedy) vs greedy

In [3]:
evaluate(env, best_candidate_policy, n=30)

{'n': 30, 'win_rate': 0.5, 'draw_rate': 0.0, 'avg_margin': 7.366666666666666}

## Train a candidate-evaluation agent (numpy REINFORCE starter)
Linear policy over per-candidate features (score, #tiles, orientation, position,
word length, **rack-leave points**). A starter to prove the loop — swap in a
neural net over the board/rack planes to actually surpass greedy.

In [ ]:
import os
from src.rl import train

agent, history = train(env, episodes=240, batch=12, lr=0.2)
print("weights:", agent.w.round(3))
print("trained vs greedy:", evaluate(env, agent.act_greedy, n=30))

# Persist so 5_play_from_screenshot.ipynb can load it.
os.makedirs("models", exist_ok=True)
agent.save("models/linear_agent.npy")
print("saved -> models/linear_agent.npy")


## Stronger: PyTorch actor-critic on random boards

The non-greedy model. Trains on a fresh randomised Wordfeud board each episode
(so it learns to *read* bonus squares, not memorise them), with richer features
(rack-leave composition + game state) and a value-function critic. Bump
`batches` up for a real run — this is a short demo.

In [ ]:
from src.boards import random_board
from src.rl_torch import train as train_torch

# Random board each episode (Wordfeud default mode).
rnd_env = WordfeudEnv(engine, board_sampler=random_board, max_candidates=48, seed=0)

print("greedy vs greedy on random boards:", evaluate(rnd_env, best_candidate_policy, n=24))

torch_agent, hist = train_torch(rnd_env, batches=40, episodes_per_batch=6, lr=2e-3)
print("torch agent vs greedy:", evaluate(rnd_env, torch_agent.act_greedy, n=24))

torch_agent.save("models/torch_agent.pt")
print("saved -> models/torch_agent.pt")